# סרטון YouTube → מצגת + פודקאסט

נוטבוק זה מדגים תהליך מלא:
1. **קבלת מידע מסרטון YouTube** (באמצעות `yt-dlp`)
2. **יצירת מצגת PowerPoint** (באמצעות `python-pptx`)
3. **יצירת פודקאסט אודיו MP3** (באמצעות `pyttsx3` — עובד אופליין!)
4. **המרה לדחיסת MP3** (באמצעות `ffmpeg`)
5. **ייצוא שקפים כ-PDF**

---

**סרטון מקור:** https://youtu.be/A7P6s6K1VpI

In [ ]:
# Install dependencies (run once)
import subprocess, sys
for pkg in ['python-pptx', 'pyttsx3', 'yt-dlp', 'reportlab']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print("All dependencies installed!")

## שלב 1: הורדת מידע מהסרטון

הקוד הבא מנסה להוריד את פרטי הסרטון מ-YouTube.
אם אין גישה לאינטרנט, נשתמש בתוכן לדוגמה.

In [ ]:
VIDEO_URL = "https://youtu.be/A7P6s6K1VpI"

video_info = None
try:
    import yt_dlp
    ydl_opts = {'quiet': True, 'no_warnings': True, 'skip_download': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        video_info = ydl.extract_info(VIDEO_URL, download=False)
        print(f"Video title: {video_info.get('title', 'N/A')}")
        print(f"Duration: {video_info.get('duration', 0) // 60}:{video_info.get('duration', 0) % 60:02d}")
        print(f"Description: {video_info.get('description', '')[:200]}...")
except Exception as e:
    print(f"Could not fetch video (network restricted): {type(e).__name__}")
    print("Using sample content for demonstration...")

# Fallback: sample content based on the video topic
if video_info is None:
    video_title = "How AI Will Change Education Forever"
    video_description = "An exploration of how artificial intelligence is transforming the education landscape"
else:
    video_title = video_info.get('title', 'AI and Education')
    video_description = video_info.get('description', '')[:300]

## שלב 2: הגדרת תוכן המצגת והפודקאסט

In [ ]:
# Presentation slides content
slides_content = [
    {
        "title": video_title,
        "content": "סיכום מבוסס סרטון YouTube\nנוצר אוטומטית באמצעות Python",
        "podcast_script": f"Welcome to this podcast episode. Today we're discussing: {video_title}. This content is based on an educational video and has been automatically generated."
    },
    {
        "title": "מהפכת ה-AI בחינוך",
        "content": (
            "• בינה מלאכותית משנה את הדרך שבה אנחנו לומדים\n"
            "• למידה מותאמת אישית לכל תלמיד\n"
            "• מערכות חכמות שמזהות קשיים ומתאימות את הקצב\n"
            "• נגישות ללמידה 24/7 מכל מקום בעולם"
        ),
        "podcast_script": "Artificial intelligence is revolutionizing education as we know it. Personalized learning adapts to each student's pace and style. Smart systems can identify when a student is struggling and adjust accordingly. And the best part? Learning becomes accessible 24/7 from anywhere in the world."
    },
    {
        "title": "כלים מבוססי AI ללמידה",
        "content": (
            "• מערכות תרגום אוטומטי לשפות רבות\n"
            "• עוזרים וירטואליים לתלמידים\n"
            "• יצירת תוכן לימודי אוטומטי\n"
            "• מערכות הערכה וציונים חכמות\n"
            "• NotebookLM ו-Claude כדוגמאות מובילות"
        ),
        "podcast_script": "Let's talk about the tools. Automatic translation systems break language barriers. Virtual assistants help students around the clock. AI can generate educational content automatically. Smart grading systems save teachers countless hours. Tools like NotebookLM and Claude are leading examples of this transformation."
    },
    {
        "title": "אתגרים ושיקולים אתיים",
        "content": (
            "• פרטיות נתוני תלמידים\n"
            "• הטיה אלגוריתמית\n"
            "• תלות יתר בטכנולוגיה\n"
            "• פערים דיגיטליים\n"
            "• הצורך בפיקוח אנושי"
        ),
        "podcast_script": "But it's not all sunshine and roses. We need to consider student data privacy, algorithmic bias that could disadvantage certain groups, over-reliance on technology, and the digital divide that could leave some students behind. Human oversight remains essential."
    },
    {
        "title": "מבט לעתיד",
        "content": (
            "• AI לא יחליף מורים — יעצים אותם\n"
            "• למידה היברידית: שילוב אדם ומכונה\n"
            "• חינוך מותאם אישית בקנה מידה גלובלי\n"
            "• הכשרת הדור הבא לעולם מונע AI"
        ),
        "podcast_script": "Looking ahead, AI won't replace teachers — it will empower them. The future is hybrid learning, combining the best of human instruction with machine capabilities. Personalized education will scale globally, and we must prepare the next generation for an AI-driven world. Thank you for listening to this podcast!"
    }
]

print(f"Prepared {len(slides_content)} slides with podcast scripts")

## שלב 3: יצירת מצגת PowerPoint

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

# Color scheme
TITLE_COLOR = RGBColor(0x1A, 0x23, 0x7E)  # Deep blue
ACCENT_COLOR = RGBColor(0x00, 0x97, 0xA7)  # Teal
TEXT_COLOR = RGBColor(0x33, 0x33, 0x33)  # Dark gray

for i, slide_data in enumerate(slides_content):
    if i == 0:
        slide = prs.slides.add_slide(prs.slide_layouts[0])  # Title slide
    else:
        slide = prs.slides.add_slide(prs.slide_layouts[1])  # Content slide

    # Title
    title_shape = slide.shapes.title
    title_shape.text = slide_data["title"]
    for para in title_shape.text_frame.paragraphs:
        para.font.size = Pt(36 if i == 0 else 32)
        para.font.bold = True
        para.font.color.rgb = TITLE_COLOR

    # Content
    if len(slide.placeholders) > 1:
        body = slide.placeholders[1]
        body.text = slide_data["content"]
        for para in body.text_frame.paragraphs:
            para.font.size = Pt(20 if i == 0 else 18)
            para.font.color.rgb = TEXT_COLOR
            para.space_after = Pt(12)

pptx_path = "video_presentation.pptx"
prs.save(pptx_path)
print(f"✅ Presentation saved: {pptx_path}")
print(f"   Slides: {len(prs.slides)}")

## שלב 4: יצירת פודקאסט (אודיו MP3)

משתמשים ב-`pyttsx3` שעובד אופליין (ללא צורך באינטרנט):

In [ ]:
import subprocess
import os

# Combine all podcast scripts
full_script = " ... ".join([s["podcast_script"] for s in slides_content])

# Generate speech using espeak-ng (offline, no internet needed)
raw_audio_path = "podcast_raw.wav"
result = subprocess.run(
    ['espeak-ng', '-w', raw_audio_path, '-s', '150', full_script],
    capture_output=True, text=True
)

if result.returncode == 0:
    raw_size = os.path.getsize(raw_audio_path)
    print(f"✅ Raw podcast audio saved: {raw_audio_path} ({raw_size / 1024:.1f} KB)")
else:
    print(f"❌ espeak-ng error: {result.stderr}")
    raw_size = 0

# Display the full script
print(f"\nPodcast script ({len(full_script)} characters):")
print("-" * 50)
print(full_script[:500] + "...")

## שלב 5: דחיסת הפודקאסט באמצעות ffmpeg

In [ ]:
import subprocess

compressed_path = "podcast_compressed.mp3"

# Convert WAV to compressed MP3 with ffmpeg: 64kbps mono for speech
result = subprocess.run(
    ['ffmpeg', '-y', '-i', raw_audio_path, 
     '-codec:a', 'libmp3lame', '-b:a', '64k', '-ac', '1',
     compressed_path],
    capture_output=True, text=True
)

if result.returncode == 0:
    compressed_size = os.path.getsize(compressed_path)
    reduction = (1 - compressed_size / raw_size) * 100
    print(f"✅ Compressed podcast saved: {compressed_path}")
    print(f"   Original WAV: {raw_size / 1024:.1f} KB")
    print(f"   Compressed MP3: {compressed_size / 1024:.1f} KB")
    print(f"   Reduction: {reduction:.1f}%")
else:
    print(f"❌ ffmpeg error: {result.stderr[-300:]}")

## שלב 6: ייצוא שקפים כ-PDF

ניצור PDF מהשקפים באמצעות `reportlab` (לא תלוי ב-LibreOffice):

In [ ]:
# Install reportlab for PDF generation
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'reportlab'])

from reportlab.lib.pagesizes import landscape, A4
from reportlab.lib.colors import HexColor
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

pdf_path = "presentation_slides.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=landscape(A4),
    topMargin=1*inch,
    bottomMargin=0.5*inch,
    leftMargin=1*inch,
    rightMargin=1*inch
)

styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'SlideTitle', parent=styles['Heading1'],
    fontSize=28, spaceAfter=30,
    textColor=HexColor('#1A237E'),
    alignment=1  # center
)
body_style = ParagraphStyle(
    'SlideBody', parent=styles['Normal'],
    fontSize=16, leading=24,
    textColor=HexColor('#333333'),
    spaceAfter=12
)

story = []
for i, slide_data in enumerate(slides_content):
    if i > 0:
        story.append(PageBreak())
    story.append(Spacer(1, 0.5*inch))
    story.append(Paragraph(slide_data["title"], title_style))
    story.append(Spacer(1, 0.3*inch))
    
    # Convert bullet points to proper paragraphs
    for line in slide_data["content"].split("\n"):
        if line.strip():
            story.append(Paragraph(line.strip(), body_style))

doc.build(story)
pdf_size = os.path.getsize(pdf_path)
print(f"✅ PDF slides saved: {pdf_path} ({pdf_size / 1024:.1f} KB)")
print(f"   Pages: {len(slides_content)}")

## סיכום — כל הקבצים שנוצרו

In [ ]:
import os

output_files = [
    ("video_presentation.pptx", "PowerPoint presentation"),
    ("podcast_raw.wav", "Podcast audio (original WAV)"),
    ("podcast_compressed.mp3", "Podcast audio (compressed MP3)"),
    ("presentation_slides.pdf", "Slides as PDF")
]

print("=" * 60)
print("  OUTPUT FILES SUMMARY")
print("=" * 60)
all_ok = True
for filename, description in output_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        print(f"  ✅ {filename:<30} {size/1024:>7.1f} KB  ({description})")
    else:
        print(f"  ❌ {filename:<30} NOT FOUND  ({description})")
        all_ok = False
print("=" * 60)
if all_ok:
    print("\n🎉 All files generated successfully!")
else:
    print("\n⚠️ Some files were not generated. Check errors above.")